# Building a Gene KO Study

## Generate Constrained Models of the STM only condition and the Mixed STM condition

In [1]:
# Set the gurobi solver 

import os
from cobra.core.configuration import Configuration

os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

# Read in the gene weights CSV files

import pandas as pd

stm_threshold_10_90 = pd.read_csv("../imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("../imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("../imat_prep_test/stm_threshold_25_75.csv")

mixed_stm_threshold_10_90 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_25_75.csv")

# Make a python set of each gene weights CSV

stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

mixed_stm_set_10_90 = set(mixed_stm_threshold_10_90["genes"].astype(str).str.strip())
mixed_stm_set_15_85 = set(mixed_stm_threshold_15_85["genes"].astype(str).str.strip())
mixed_stm_set_25_75 = set(mixed_stm_threshold_25_75["genes"].astype(str).str.strip())

# QC: Check the count of genes
print("STM1090 gene count",len(stm_set_10_90), "|", "STM1585 gene count", len(stm_set_15_85), "|", "STM2575 gene count", len(stm_set_25_75))
print("Mixed STM1090 gene count",len(mixed_stm_set_10_90), "|", "Mixed STM1585 gene count", len(mixed_stm_set_15_85), "|", "Mixed STM2575 gene count", len(mixed_stm_set_25_75))

# Load STM_V1_0

from cobra.io import load_model, load_json_model

control_model = load_model("STM_v1_0")

# Save all genes in the control model in a set

model_genes = set([g.id for g in control_model.genes])

# QC: Check the count of genes
print("STM_V1_0 Control Model has", len(model_genes), "total genes")

# Subset the gene weights lists for each sample to keep only the genes also found in the control model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

mixed_stm1090_model = mixed_stm_threshold_10_90[mixed_stm_threshold_10_90["genes"].isin(model_genes)]
mixed_stm1585_model = mixed_stm_threshold_15_85[mixed_stm_threshold_15_85["genes"].isin(model_genes)]
mixed_stm2575_model = mixed_stm_threshold_25_75[mixed_stm_threshold_25_75["genes"].isin(model_genes)]

# Align the gene weights lists to the control model genes list and assign missing genes a weight of 0

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

subset_weights_mixedstm1090 = mixed_stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm1585 = mixed_stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm2575 = mixed_stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)


# QC: Confirm all samples have the same amount of gene weights

# 1. Make a list of the subset gene weights
subset_gene_weights = [
    subset_weights_stm1090,
    subset_weights_stm1585,
    subset_weights_stm2575,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
gene_counts = [len(weights) for weights in subset_gene_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(gene_counts)) == 1:
    print(f"All models have {gene_counts[0]} genes")
else:
    print("Gene counts differ between models")
    print(gene_counts)

# Convert gene weights to reaction weights with iMAT

from imatpy.parse_gpr import gene_to_rxn_weights

rxn_weights_stm1090 = gene_to_rxn_weights(control_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(control_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(control_model, subset_weights_stm2575)

rxn_weights_mixedstm1090 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1090)
rxn_weights_mixedstm1585 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1585)
rxn_weights_mixedstm2575 = gene_to_rxn_weights(control_model, subset_weights_mixedstm2575)

# QC: Confirm all samples have the same amount of reaction weights

# 1. Make a list of the reaction weights
reaction_weights = [
    rxn_weights_stm1090,
    rxn_weights_stm1585,
    rxn_weights_stm2575,
    rxn_weights_mixedstm1090,
    rxn_weights_mixedstm1585,
    rxn_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
reactions_counts = [len(weights) for weights in reaction_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(reactions_counts)) == 1:
    print(f"All models have {reactions_counts[0]} reactions")
else:
    print("Reaction counts differ between models")
    print(reactions_counts)

# Generate constrained models with iMAT by integrating the reaction weights

from imatpy.model_creation import generate_model

stm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

mixedstm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1090, method = "imat_restrictions")
mixedstm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1585, method = "imat_restrictions")
mixedstm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm2575, method = "imat_restrictions")

STM1090 gene count 4671 | STM1585 gene count 4671 | STM2575 gene count 4671
Mixed STM1090 gene count 4671 | Mixed STM1585 gene count 4671 | Mixed STM2575 gene count 4671
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu
STM_V1_0 Control Model has 1271 total genes
All models have 1271 genes
All models have 2545 reactions
Read LP format model from file /tmp/tmpaezzgigu.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpk2klpk7j.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmptsdcyw01.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp5k19he7u.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpjqbfy3up.lp
Reading time 

## Dictionary of Models Generated

In [2]:
# make a dictionary of all the models 

models = {
    "stm_unconstrained": control_model,
    "mixed_stm1090": mixedstm1090_model,
    "mixed_stm1585": mixedstm1585_model,
    "mixed_stm2575": mixedstm2575_model,
    "stm_1090": stm1090_model,
    "stm_1585": stm1585_model,
    "stm_2575": stm2575_model
}

## Defining the Mouse Minimal Medium Conditions

What are the units for the value for each exchange reaction? How do we decide on a number when simulating for all exchange reactions? 
add tetrathionate to the minimal media and keep nitrate becuase it's mostly host derived. 

In [3]:
# Make a copy of the default medium from stm_v1_0
default_medium = dict(models["stm_unconstrained"].medium)

# Add mouse-derived, gut metabolites to the default medium to create a mouse gut minimal medium
mouse_min_medium = dict(default_medium)
mouse_min_medium.update({
    "EX_o2_e": 0.5, # updating to anoxic conditions
    "EX_no3_e": 0.5, # should we remove this?
    "EX_for_e": 4.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_glc__D_e": 0.0 # removing from default medium
})

In [4]:
# Add mouse-derived and host-derived gut metabolites to the default medium to create a mouse gut infection medium
infection_medium = dict(default_medium)
infection_medium.update({
    "EX_o2_e": 0.0, # removing from default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

## Simulating Deletions

**Notes from 10/24/26**

ToDos:
- Simulate with minimal mouse media 
- Simulate with mouse gut environment media (host and microbe derived metabolites)
- Use case growth rate for each model with and without a deletion of a: single gene, a single reaction, double gene, double reaction. 


What I want to explore: 
- FVA vs FBA results output. 
- Solution flux, summary flux, nadh summary, atpsummary , and media metabolites summary for each deletion model simulation.
- what is a production envelope. 

For the meeting with Titus on 10/27/26, I want to have all the deletion simulations done because what I'll need help with is data analysis. 



In [5]:
import pandas as pd 
from time import time 

from cobra.flux_analysis import (
    single_gene_deletion, single_reaction_deletion, double_gene_deletion, double_reaction_deletion)

### Single gene deletions

In [6]:
# copy a model 
#stm1090_geneKO = models["stm_1090"].copy()

# assign the minimal mouse gut medium to stm1090
#stm1090_geneKO.medium = mouse_min_medium

# perform all single gene deletions on stm1090
#stm1090_gene_deletions = single_gene_deletion(stm1090_geneKO, stm1090_geneKO.genes[:10])

In [7]:
#stm1090_gene_deletions

### Single reaction deletions

In [8]:
# copy a model 
#stm1090_rxnKO = models["stm_1090"].copy()

# assign the minimal mouse gut medium to stm1090
#stm1090_rxnKO.medium = mouse_min_medium

# perform all single gene deletions on stm1090
#stm1090_rxnKO_deletions = single_reaction_deletion(stm1090_rxnKO, stm1090_rxnKO.reactions[:10])

In [9]:
#stm1090_rxnKO_deletions

### Double gene deletions

In [10]:
# copy a model 
#stm1090_doublegeneKO = models["stm_1090"].copy()

# assign the minimal mouse gut medium to stm1090
#stm1090_doublegeneKO.medium = mouse_min_medium

# perform all double gene deletions on stm1090
#stm1090_doublegene_deletions = double_gene_deletion(stm1090_doublegeneKO, stm1090_doublegeneKO.genes[-3:])

In [11]:
#stm1090_doublegene_deletions

### Perform all double reaction deletions

In [12]:
# copy a model 
#stm1090_doublerxnKO = models["stm_1090"].copy()

# assign the minimal mouse gut medium to stm1090
#stm1090_doublerxnKO.medium = mouse_min_medium

# perform all double reaction deletions on stm1090
#stm1090_doublerxn_deletions = double_reaction_deletion(stm1090_doublerxnKO, stm1090_doublerxnKO.reactions[-3:])

In [13]:
#stm1090_doublerxn_deletions

## How Extract Results from KO Simulations

I am going to write a script for each simulation type: single gene, single reaction, double gene, and double reaction. The output for each needs to be results from each solved model so I have to determine what data I am collecting. 

In [14]:
#print('The type for the output of gene deletions is:', type(stm1090_gene_deletions))

In [15]:
#help(single_gene_deletion)

'single_gene_deletion` returns a pandas dataframe with the growth of the model and id for the gene knocked out. The solution flux for a model is not saved in this method. Without the model saved, we can't access metabolite uptake and secretion or flux through pathways. 

## Building a for loop for single gene deletions in all models 
10.29.26 and 10.30.26

In [16]:
geneko_results_list = [ ]

for name, model in models.items():

    geneko_model = model.copy()
    geneko_medium = infection_medium.copy()
    geneko_model.medium = geneko_medium

    geneko_results = single_gene_deletion(geneko_model, geneko_model.genes, method="fba")

    geneko_results['model'] = name
    geneko_results_list.append(geneko_results)
    geneko_results_df = pd.concat(geneko_results_list)
    geneko_results_df.to_csv(f"geneko_results.csv", index=False)

    break

Read LP format model from file /tmp/tmphsciym8a.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


In [17]:
print(geneko_results_list)

[            ids    growth   status              model
0     {STM1193}  0.875012  optimal  stm_unconstrained
1     {STM3513}  0.876923  optimal  stm_unconstrained
2     {STM1954}  0.876923  optimal  stm_unconstrained
3     {STM0532}  0.876923  optimal  stm_unconstrained
4     {STM0757}  0.876923  optimal  stm_unconstrained
...         ...       ...      ...                ...
1266  {STM2952}  0.524277  optimal  stm_unconstrained
1267  {STM4220}  0.876923  optimal  stm_unconstrained
1268  {STM3539}  0.000000  optimal  stm_unconstrained
1269  {STM0417}  0.000000  optimal  stm_unconstrained
1270  {STM0965}  0.876923  optimal  stm_unconstrained

[1271 rows x 4 columns]]


`model.genes` returns a list of gene id's. I need to figure out how to get the gene name.

My idea is to pass each gene id in the geneko_results_list to `model.genes.[id]' to get the gene name, then add the name to a new dataframe.

In [18]:
models["stm_1090"].genes.get_by_id("STM2646").name

'yfiD'

Pass each gene id value from the id column of the `geneko_results` dataframe through `geneko_model.genes.get_by_id[""].name`. Create a new column in `geneko_results` and append the name to the column. 

In [19]:
df = geneko_results_list[0]
df

,ids,growth,status,model
0,{STM1193},0.875012,optimal,stm_unconstrained
1,{STM3513},0.876923,optimal,stm_unconstrained
2,{STM1954},0.876923,optimal,stm_unconstrained
3,{STM0532},0.876923,optimal,stm_unconstrained
4,{STM0757},0.876923,optimal,stm_unconstrained
...,...,...,...,...
1266,{STM2952},0.524277,optimal,stm_unconstrained
1267,{STM4220},0.876923,optimal,stm_unconstrained
1268,{STM3539},0.000000,optimal,stm_unconstrained
1269,{STM0417},0.000000,optimal,stm_unconstrained


Get the data type for each value of the dataframe. 

In [20]:
print(df["ids"].dtype)

object


Write a lambda function that takes the gene id from the ids dataframe and gets the gene id. 

In [21]:
df["Gene Name"] = df["ids"].map(
lambda gene: geneko_model.genes.get_by_id(gene).name
)

TypeError: unhashable type: 'set'

In [ ]:
print(df)

Write a function to get the gene name from the `geneko_results_list`. Use this in place of a lambda function. 

In [ ]:
#def get_gene_name(

The idea here is that the gene id's in the `ids` column of the dataframe are actually objects so they need to be converted to strings for the `get_by_id` function. 

The issue is that when using `.astype(str)` the id's are converted to {STM0001} when the id is actually STM0001, no curly brace. Secondly, I need to check in the cobrapy documentation if there's such a thing as a gene object and if that's the data type of the gene knockout method in the returned dataframe. 

In [22]:
df["Gene Name"] = df["ids"].astype(str).map(
lambda gene: geneko_model.genes.get_by_id(gene).name
)

KeyError: "{'STM1193'}"

### 11/01/26 

The issue above is that `single_gene_deletion()` stores the gene id as objects and not strings. The `ids` column stores set-like objects like `{'STM1193'}` so when I tried `astype(str)` it converted `"{'STM1193'}"` to a string which does not match the gene id "STM1193" resulting in a keyerror. 

`next(iter())` creates an iterator over the set in the `ids` column which returns a string of the gene id and not an object. I don't know exactly how this works. I don't think this will work for two gene names because the 'ids' column will have two gene id objects that `next(iter())` only extracts one object. 

In [ ]:
geneko_results_list = []

for name, model in models.items():

    geneko_model = model.copy()
    geneko_medium = infection_medium.copy()
    geneko_model.medium = geneko_medium

    geneko_results = single_gene_deletion(
        geneko_model,
        geneko_model.genes,
        method="fba"
    )

    # Extract gene ID from the set
    geneko_results["Gene ID"] = geneko_results["ids"].map(
        lambda genes: next(iter(genes))
    )

    # Get gene name from model
    geneko_results["Gene Name"] = geneko_results["Gene ID"].map(
        lambda gene_id: geneko_model.genes.get_by_id(gene_id).name
    )

    geneko_results["model"] = name

    geneko_results_list.append(geneko_results)

geneko_results_df = pd.concat(geneko_results_list, ignore_index=True)

geneko_results_df.to_csv("geneko_results.csv", index=False)

Read LP format model from file /tmp/tmpcbt16unb.lp
Reading time = 1.54 seconds
: 1802 rows, 5090 columns, 19612 nonzeros

Interrupt request received


Exception in thread Exception in threading.excepthook:
Exception ignored in thread started by: <bound method Thread._bootstrap of <Thread(Thread-11 (_handle_workers), stopped daemon 12462096172608)>>
Traceback (most recent call last):
  File "/home/gmvaz/Installs/miniforge3/envs/gems/lib/python3.11/threading.py", line 1002, in _bootstrap
    self._bootstrap_inner()
  File "/home/gmvaz/Installs/miniforge3/envs/gems/lib/python3.11/threading.py", line 1047, in _bootstrap_inner
    self._invoke_excepthook(self)
  File "/home/gmvaz/Installs/miniforge3/envs/gems/lib/python3.11/threading.py", line 1359, in invoke_excepthook
    local_print("Exception in threading.excepthook:",
  File "/home/gmvaz/Installs/miniforge3/envs/gems/lib/python3.11/site-packages/ipykernel/iostream.py", line 670, in flush
    self.pub_thread.schedule(self._flush)
  File "/home/gmvaz/Installs/miniforge3/envs/gems/lib/python3.11/site-packages/ipykernel/iostream.py", line 339, in schedule
    self._event_pipe.send(b"")
 


Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request received

Interrupt request r

KeyboardInterrupt: 


Interrupt request received
